# Construction Materials Cost Planner - Analysis Notebook

**Author:** Mohab Nasser Ahmed

This notebook walks through the analysis performed by the two application
modules and records a short test plan (3 valid + 3 invalid cases) with the
expected and actual results.

**Modules**
1. `material_calculator.py` - concrete element model, volume and material
   quantity calculations.
2. `cost_analysis.py` - cleaning, merging, cost analysis, a NumPy price
   what-if, and the four charts.


In [1]:
import os
import pandas as pd
import numpy as np

from material_calculator import ConcreteElement, ElementManager, ValidationError, load_elements_csv
import cost_analysis as ca

DATA = "data"
pd.set_option("display.width", 120)

## 1. Load elements and compute required quantities

In [2]:
mgr = load_elements_csv(os.path.join(DATA, "elements.csv"))
quantities = pd.DataFrame([e.quantities() for e in mgr.list_elements()])
quantities

,element_id,element_name,site,length_m,width_m,height_m,waste_pct,net_volume_m3,volume_with_waste_m3,cement_kg,sand_m3,aggregate_m3
0,E01,Footing F1,Site A,2.5,2.50,0.60,5.0,3.750,3.9375,1181.25,1.9688,3.1500
1,E02,Column C1,Site A,0.4,0.40,3.00,4.0,0.480,0.4992,149.76,0.2496,0.3994
2,E03,Beam B1,Site A,6.0,0.30,0.50,5.0,0.900,0.9450,283.50,0.4725,0.7560
3,E04,Slab S1,Site A,8.0,5.00,0.15,6.0,6.000,6.3600,1908.00,3.1800,5.0880
4,E05,Footing F2,Site B,3.0,3.00,0.70,5.0,6.300,6.6150,1984.50,3.3075,5.2920
5,E06,Wall W1,Site B,10.0,0.25,3.00,7.0,7.500,8.0250,2407.50,4.0125,6.4200
6,E07,Slab S2,Site B,12.0,6.00,0.20,6.0,14.400,15.2640,4579.20,7.6320,12.2112
7,E08,Column C2,Site B,0.5,0.50,3.50,4.0,0.875,0.9100,273.00,0.4550,0.7280


The required quantities follow the assumed mix rates applied to the
*volume including waste*: 300 kg cement, 0.5 m3 sand and 0.8 m3 aggregate per m3.

## 2. Clean the purchase and price data

In [3]:
prices = ca.load_prices(os.path.join(DATA, "prices.csv"))
purchases = ca.load_purchases(
    os.path.join(DATA, "purchases.csv"),
    valid_elements={e.element_id for e in mgr.list_elements()},
    valid_suppliers=set(prices["supplier_id"]),
)
print("Clean price rows:", len(prices), "| clean purchase rows:", len(purchases))

  [purchases] 40 -> 38 (dup rows) -> 35 (bad qty) -> 35 (dup IDs) -> 35 (unknown refs)
Clean price rows: 6 | clean purchase rows: 35


Cleaning removes exactly duplicated rows (e.g. T009, T025), rows with a
missing or non-positive quantity (T012, T014, T027), duplicate transaction IDs,
and any purchase referencing an unknown element or supplier.

## 3. Merge and compute costs per site, supplier and month

In [4]:
merged = ca.build_cost_table(purchases, prices, mgr)
print("By site");     print(ca.cost_by(merged, "site").to_string(index=False))
print("\nBy supplier"); print(ca.cost_by(merged, "supplier_id").to_string(index=False))
print("\nBy month");    print(ca.cost_by(merged, "month").to_string(index=False))

By site
  site    cost
Site A  712.32
Site B 1608.02

By supplier
supplier_id    cost
        S01 1668.00
        S02  652.35

By month
  month    cost
2026-08  699.85
2026-09 1620.50


## 4. Required vs purchased (shortage / surplus)

In [5]:
comparison = ca.compare_required_vs_purchased(mgr, merged)
comparison

,element_id,site,material,unit,required_qty,purchased_qty,balance,status
0,E01,Site A,Aggregate,m3,3.1500,3.35,0.2000,SURPLUS
1,E01,Site A,Cement,kg,1181.2500,1040.00,-141.2500,SHORTAGE
2,E01,Site A,Sand,m3,1.9688,2.25,0.2812,SURPLUS
3,E02,Site A,Aggregate,m3,0.3994,0.25,-0.1494,SHORTAGE
4,E02,Site A,Cement,kg,149.7600,70.00,-79.7600,SHORTAGE
5,E02,Site A,Sand,m3,0.2496,0.25,0.0004,SURPLUS
6,E03,Site A,Aggregate,m3,0.7560,0.85,0.0940,SURPLUS
7,E03,Site A,Cement,kg,283.5000,290.00,6.5000,SURPLUS
8,E03,Site A,Sand,m3,0.4725,0.45,-0.0225,SHORTAGE
9,E04,Site A,Aggregate,m3,5.0880,4.15,-0.9380,SHORTAGE


In [6]:
# Summary of shortages vs surpluses
comparison["status"].value_counts()

status
SURPLUS     12
SHORTAGE    12
Name: count, dtype: int64

## 5. NumPy what-if: raise one material's price

In [7]:
sim = ca.simulate_price_increase(mgr, prices, "Cement", 10)
pd.DataFrame({
    "material": sim["materials"],
    "required_qty": sim["required_qty"],
    "baseline_price": sim["baseline_unit_price"],
    "new_price": sim["new_unit_price"],
    "cost_before": sim["cost_before"],
    "cost_after": sim["cost_after"],
})

,material,required_qty,baseline_price,new_price,cost_before,cost_after
0,Cement,12766.7100,0.11,0.121,1404.34,1544.77
1,Sand,21.2778,14.50,14.500,308.53,308.53
2,Aggregate,34.0446,21.00,21.000,714.94,714.94


In [8]:
print(f"Total required-quantity cost before: {sim['total_before']}")
print(f"Total required-quantity cost after : {sim['total_after']}")
print(f"Delta from +10% on Cement          : {sim['total_delta']}")

Total required-quantity cost before: 2427.8
Total required-quantity cost after : 2568.24
Delta from +10% on Cement          : 140.43


## 6. Charts\nFour charts are written to the `charts/` folder (line, bar, scatter, histogram).

In [9]:
paths = ca.make_charts(merged, comparison, "charts")
paths

['charts/line_monthly_cost.png',
 'charts/bar_cost_per_site.png',
 'charts/scatter_qty_vs_cost.png',
 'charts/hist_line_costs.png']

## 7. Test plan - 3 valid and 3 invalid cases

Each test records the **expected** and **actual** result. A small helper prints
a PASS/FAIL line so the outcomes are easy to read.

In [10]:
def check(name, expected, actual):
    ok = "PASS" if expected == actual else "FAIL"
    print(f"[{ok}] {name}\n      expected: {expected}\n      actual:   {actual}")

results = []

### Valid cases

In [11]:
# V1 - Net volume of a 2 x 3 x 0.5 element = 3.0 m3
e = ConcreteElement("TV1", "Test", "Site A", 2.0, 3.0, 0.5, 0.0)
check("V1 net volume (no waste)", 3.0, round(e.net_volume, 4))

# V2 - Volume including 10% waste = 3.0 * 1.10 = 3.3 m3
e2 = ConcreteElement("TV2", "Test", "Site A", 2.0, 3.0, 0.5, 10.0)
check("V2 volume with 10% waste", 3.3, round(e2.volume_with_waste, 4))

# V3 - Cement for 3.3 m3 (incl. waste) = 3.3 * 300 = 990 kg
check("V3 cement quantity", 990.0, round(e2.cement_kg, 4))

[PASS] V1 net volume (no waste)
      expected: 3.0
      actual:   3.0
[PASS] V2 volume with 10% waste
      expected: 3.3
      actual:   3.3
[PASS] V3 cement quantity
      expected: 990.0
      actual:   990.0


### Invalid cases (each should raise `ValidationError`)

In [12]:
def expect_error(name, func):
    try:
        func()
        check(name, "ValidationError", "no error raised")
    except ValidationError as exc:
        check(name, "ValidationError", "ValidationError")

# I1 - zero dimension is rejected
expect_error("I1 zero height rejected",
             lambda: ConcreteElement("TI1", "Bad", "Site A", 2.0, 3.0, 0.0, 5.0))

# I2 - waste percentage above 100 is rejected
expect_error("I2 waste > 100 rejected",
             lambda: ConcreteElement("TI2", "Bad", "Site A", 2.0, 3.0, 0.5, 150.0))

# I3 - duplicate element_id is rejected by the manager
def dup():
    m = ElementManager()
    m.add(ConcreteElement("D1", "A", "Site A", 1, 1, 1, 0))
    m.add(ConcreteElement("D1", "B", "Site A", 1, 1, 1, 0))
expect_error("I3 duplicate element_id rejected", dup)

[PASS] I1 zero height rejected
      expected: ValidationError
      actual:   ValidationError
[PASS] I2 waste > 100 rejected
      expected: ValidationError
      actual:   ValidationError
[PASS] I3 duplicate element_id rejected
      expected: ValidationError
      actual:   ValidationError


### Test results summary

| # | Case | Type | Expected | Result |
|---|------|------|----------|--------|
| V1 | Net volume 2x3x0.5 | valid | 3.0 m3 | PASS |
| V2 | Volume with 10% waste | valid | 3.3 m3 | PASS |
| V3 | Cement for 3.3 m3 | valid | 990 kg | PASS |
| I1 | Zero height | invalid | ValidationError | PASS |
| I2 | Waste > 100 | invalid | ValidationError | PASS |
| I3 | Duplicate element_id | invalid | ValidationError | PASS |
